<a href="https://colab.research.google.com/github/maminsed/Cool-Stuff/blob/colab_branch/MIT_ML/tf_music_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install comet_ml > /dev/null 2>&1
import comet_ml
from google.colab import userdata
COMET_API_KEY = userdata.get('COMMET_API_KEY')

!pip install mitdeeplearning --quiet
import mitdeeplearning as mdl
import tensorflow as tf
import numpy as np
import os
import tqdm

!apt-get install abcmidi timidity > /dev/null 2>&1

assert COMET_API_KEY != "", "fill out the COMET_API_KEY"
assert len(tf.config.list_physical_devices('GPU')) > 0, "use GPU"

In [ ]:
# Download the dataset
songs = mdl.lab1.load_training_data()
# Play the song
mdl.lab1.play_song(example_song)

# Perform some simple tests to make sure your batch function is working properly!
test_args = (vectorized_songs, 10, 2)
if not mdl.lab1.test_batch_func_types(get_batch, test_args) or \
   not mdl.lab1.test_batch_func_shapes(get_batch, test_args) or \
   not mdl.lab1.test_batch_func_next_step(get_batch, test_args):
   print("======\n[FAIL] could not pass tests")
else:
   print("======\n[PASS] passed all tests!")

# Checkpoint location:
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "my_ckpt.weights.h5")
os.makedirs(checkpoint_dir, exist_ok=True)


### Create a Comet experiment to track our training run ###
def create_experiment():
  # end any prior experiments
  if 'experiment' in locals():
    experiment.end()

  # initiate the comet experiment for tracking
  experiment = comet_ml.Experiment(
                  api_key=COMET_API_KEY,
                  project_name="6S191_Lab1_Part2")
  # log our hyperparameters, defined above, to the experiment
  for param, value in params.items():
    experiment.log_parameter(param, value)
  experiment.flush()

  return experiment
experiment.log_metric('...')
experiment.flush()
experiment.end()

### Play back generated songs ###

generated_songs = mdl.lab1.extract_song_snippet(generated_text)

for i, song in enumerate(generated_songs):
  # Synthesize the waveform from a song
  waveform = mdl.lab1.play_song(song)

  # If its a valid song (correct syntax), lets play it!
  if waveform:
    print("Generated song", i)
    ipythondisplay.display(waveform)

    numeric_data = np.frombuffer(waveform.data, dtype=np.int16)
    wav_file_path = f"output_{i}.wav"
    write(wav_file_path, 88200, numeric_data)

    # save your song to the Comet interface -- you can access it there
    experiment.log_asset(wav_file_path)

In [ ]:
# Download the dataset
songs = mdl.lab1.load_training_data()

# Print one of the songs to inspect it in greater detail!
example_song = songs[0]
print("\nExample song: ")
print(example_song)

In [ ]:
# Play the song
mdl.lab1.play_song(example_song)

In [ ]:
all_songs = "\n\n".join(songs)

idx2char = sorted(set(all_songs))
print(idx2char)
char2idx = {v:i for i,v in enumerate(idx2char)}

In [ ]:
def vectorize(string):
  return np.array([char2idx[s] for s in string])

In [ ]:
vectorized_songs = vectorize(all_songs)
assert isinstance(vectorized_songs,np.ndarray), "it should be a numpy array"

In [ ]:
def get_batch(vectorized_songs, seq_len, batch_size):
  size = len(vectorized_songs)
  indecies = np.random.choice(size-seq_len-1,size=batch_size)

  # gettings the input and output
  input = [vectorized_songs[i:i+seq_len] for i in indecies]
  output = [vectorized_songs[i+1:i+seq_len+1] for i in indecies]


  input = np.reshape(input,[batch_size,seq_len])
  output = np.reshape(output,[batch_size,seq_len])

  return input,output

In [ ]:
# Perform some simple tests to make sure your batch function is working properly!
test_args = (vectorized_songs, 10, 2)
if not mdl.lab1.test_batch_func_types(get_batch, test_args) or \
   not mdl.lab1.test_batch_func_shapes(get_batch, test_args) or \
   not mdl.lab1.test_batch_func_next_step(get_batch, test_args):
   print("======\n[FAIL] could not pass tests")
else:
   print("======\n[PASS] passed all tests!")

In [ ]:
def unvectorize(arr):
  return "".join([idx2char[i] for i in arr])

In [ ]:
def LSTM(rnn_units:int):
  return tf.keras.layers.LSTM(
      rnn_units,
      return_sequences=True,
      recurrent_initializer='glorot_uniform', #Might want to comment this out.
      stateful=True, # Also might want to comment this out.
      recurrent_activation='sigmoid',
      activation='tanh',
  )

In [ ]:
def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
  return tf.keras.Sequential([
      tf.keras.layers.Embedding(vocab_size,embedding_dim),
      LSTM(rnn_units),
      tf.keras.layers.Dense(vocab_size),
  ])

In [ ]:
vocab_size = len(idx2char)
model = build_model(vocab_size, 256,1024,32)
model.build((32,None))
model.summary()

In [ ]:
def compute_loss(labels,logits):
  return tf.keras.losses.sparse_categorical_crossentropy(labels,logits,from_logits=True)

## predict from an untrained model

In [ ]:
model = build_model(vocab_size, 256,1024,batch_size=1)
X,Y = get_batch(vectorized_songs,100,1)
predictions = model.predict(Y)
print(f"input_shape: {X.shape}: #batch_size, seq_length")
print(f"output_shape: {predictions.shape}: #batch_size, seq_length,vocab_size")

In [ ]:
sampeled_indecies = tf.random.categorical(predictions[0],num_samples=1)
unvectorize(tf.squeeze(sampeled_indecies).numpy())

In [ ]:
print(compute_loss(Y,predictions).numpy().mean())

In [ ]:
# Checkpoint location:
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "my_ckpt.weights.h5")
os.makedirs(checkpoint_dir, exist_ok=True)

In [ ]:
vocab_size = len(idx2char)
params = dict(
  num_training_iterations = 3000,  # Increase this to train longer
  batch_size = 8,  # Experiment between 1 and 64
  seq_length = 100,  # Experiment between 50 and 500
  learning_rate = 5e-3,  # Experiment between 1e-5 and 1e-1
  embedding_dim = 256,
  rnn_units = 1024,  # Experiment between 1 and 2048
)

In [ ]:
### Create a Comet experiment to track our training run ###
def create_experiment():
  # end any prior experiments
  if 'experiment' in locals():
    experiment.end()

  # initiate the comet experiment for tracking
  experiment = comet_ml.Experiment(
                  api_key=COMET_API_KEY,
                  project_name="6S191_Lab1_Part2")
  # log our hyperparameters, defined above, to the experiment
  for param, value in params.items():
    experiment.log_parameter(param, value)
  experiment.flush()

  return experiment

In [ ]:
model = build_model(vocab_size,params['embedding_dim'],params['rnn_units'],params['batch_size'])
optimizer =  tf.keras.optimizers.Adam(params['learning_rate'])

def train_step(X,Y):
  with tf.GradientTape() as tape:
    pred = model.predict(X)
    loss = compute_loss(Y,pred)
  grads = tape.gradient(loss,model.trainable_weights)
  optimizer.apply_gradients(zip(grads,model.trainable_weights))
  return loss

history = []
plotter = mdl.util.PeriodicPlotter(sec=2, xlabel='Iterations', ylabel='Loss')
experiment = create_experiment()
if hasattr(tqdm, '_instances'): tqdm._instances.clear() # clear if it exists
for iter in tqdm(range(params["num_training_iterations"])):

  # Grab a batch and propagate it through the network
  x_batch, y_batch = get_batch(vectorized_songs, params["seq_length"], params["batch_size"])
  loss = train_step(x_batch, y_batch)

  # log the loss to the Comet interface! we will be able to track it there.
  experiment.log_metric("loss", loss.numpy().mean(), step=iter)
  # Update the progress bar and also visualize within notebook
  history.append(loss.numpy().mean())
  plotter.plot(history)

  # Update the model with the changed weights!
  if iter % 100 == 0:
    model.save_weights(checkpoint_prefix)

# Save the trained model and the weights
model.save_weights(checkpoint_prefix)
experiment.flush()